# 6.9 · 核 PCA / Kernel PCA

> **课程定位 / Where this fits**
> PCA(6.8)只能找**线性**子空间, 对弯曲流形(瑞士卷、同心圆)无能为力。核 PCA 用**核技巧**(5.5 SVM 见过): 把数据隐式映射到高维特征空间, 在那里做线性 PCA, 等价于在原空间做**非线性**降维。能"展开"PCA 展不开的非线性结构。
> Kernel PCA applies the kernel trick to PCA — implicitly mapping to a high-dimensional space and doing linear PCA there, which is nonlinear PCA in the original space.

> 💡 **面试相关 / Interview-relevant**
> - "核 PCA 与 PCA 区别 / 何时用" ★★★★★
> - "核技巧为什么避免显式映射" ★★★★★（同 SVM）
> - "为什么要中心化核矩阵(Gram matrix centering)" ★★★★
> - "核 PCA 的缺点(O(n²) 核矩阵, 无显式逆变换)" ★★★

---

## 学习目标 / Learning Objectives
1. 核 PCA 的思路: 核矩阵特征分解。
2. 核矩阵中心化的必要性。
3. 在非线性数据(同心圆/瑞士卷)上对比 PCA。
4. 不同核 + gamma 的影响, 与缺点。

## 目录 / TOC
1. [从 PCA 到核 PCA ⭐](#1)
2. [⭕ 数据: 同心圆 + 对比 PCA ⭐](#2)
3. [瑞士卷展开 + 核/gamma](#3)
4. [缺点](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 从 PCA 到核 PCA ⭐ / From PCA to Kernel PCA

普通 PCA 对协方差 $\frac1n\mathbf{X}^\top\mathbf{X}$ 做特征分解。核 PCA 想在映射 $\phi(\mathbf{x})$ 的空间里做 PCA, 但 $\phi$ 可能无限维, 不能显式算。

**核技巧**(5.5): 一切只用到内积 $\phi(\mathbf{x}_i)^\top\phi(\mathbf{x}_j)=K(\mathbf{x}_i,\mathbf{x}_j)$。可以证明, 在特征空间做 PCA 等价于对 **核矩阵(Gram 矩阵)** $\mathbf{K}$($K_{ij}=K(\mathbf{x}_i,\mathbf{x}_j)$)做特征分解:
$$\mathbf{K}\,\boldsymbol\alpha_k = \lambda_k\,\boldsymbol\alpha_k$$
特征向量 $\boldsymbol\alpha_k$ 给出每个点在第 $k$ 个非线性主成分上的投影(归一化后)。常用 RBF 核 $\exp(-\gamma\|\mathbf{x}_i-\mathbf{x}_j\|^2)$。

**关键: 核矩阵必须中心化**。因为我们无法显式地"减均值"特征空间里的点, 改为在核矩阵上做等价中心化:
$$\tilde{\mathbf{K}} = \mathbf{K} - \mathbf{1}_n\mathbf{K} - \mathbf{K}\mathbf{1}_n + \mathbf{1}_n\mathbf{K}\mathbf{1}_n$$
($\mathbf{1}_n$ 是全 $1/n$ 矩阵)。sklearn 自动处理。


<a id="2"></a>
## 2. 数据: 同心圆 + 对比 PCA ⭐ / Concentric Circles vs PCA

同心圆(6.7 用过): 线性 PCA 无论怎么投都分不开内外环, 核 PCA(RBF)能把它们在新空间里拉成线性可分。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_circles
from sklearn.decomposition import PCA, KernelPCA
sns.set_theme(style="whitegrid")

X, y = make_circles(400, factor=0.3, noise=0.05, random_state=0)

pca = PCA(2).fit_transform(X)
kpca = KernelPCA(n_components=2, kernel="rbf", gamma=10).fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", s=15); axes[0].set_title("原始: 同心圆(线性不可分)")
axes[1].scatter(pca[:,0], pca[:,1], c=y, cmap="coolwarm", s=15); axes[1].set_title("线性 PCA: 仍缠在一起")
axes[2].scatter(kpca[:,0], kpca[:,1], c=y, cmap="coolwarm", s=15); axes[2].set_title("核 PCA (RBF): 两环线性可分")
for a in axes: a.set_xlabel("分量1"); a.set_ylabel("分量2")
plt.tight_layout(); plt.show()
print("线性 PCA 无法分开同心圆(没有能分开的线性方向);")
print("核 PCA(RBF) 在隐式高维空间做 PCA → 原空间的非线性结构被展开成线性可分")


<a id="3"></a>
## 3. 瑞士卷展开 + 核/gamma / Swiss Roll & Kernels


In [ ]:
from sklearn.datasets import make_swiss_roll
from mpl_toolkits.mplot3d import Axes3D  # noqa

Xs, color = make_swiss_roll(1000, noise=0.05, random_state=0)
print(f"瑞士卷: {Xs.shape} (3D 上的 2D 弯曲流形)")
fig = plt.figure(figsize=(13, 4))
ax = fig.add_subplot(131, projection="3d")
ax.scatter(Xs[:,0], Xs[:,1], Xs[:,2], c=color, cmap="Spectral", s=8)
ax.set_title("瑞士卷 (3D)")

ax2 = fig.add_subplot(132)
p = PCA(2).fit_transform(Xs)
ax2.scatter(p[:,0], p[:,1], c=color, cmap="Spectral", s=8); ax2.set_title("线性 PCA: 拍扁但颜色仍混叠")

ax3 = fig.add_subplot(133)
kp = KernelPCA(2, kernel="rbf", gamma=0.05).fit_transform(Xs)
ax3.scatter(kp[:,0], kp[:,1], c=color, cmap="Spectral", s=8); ax3.set_title("核 PCA (RBF)")
plt.tight_layout(); plt.show()

# gamma 影响: 用"在 kPCA 嵌入上线性分类的可分性"衡量 / gamma sensitivity via linear separability
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
print("\n核 PCA 对 gamma 敏感(在同心圆 kPCA 嵌入上做线性分类的 CV 准确率):")
for g in [0.1, 1, 10, 100]:
    kp = KernelPCA(2, kernel="rbf", gamma=g).fit_transform(X)
    acc = cross_val_score(LogisticRegression(), kp, y, cv=5).mean()
    print(f"  gamma={g:>4}: 线性可分性(CV acc) {acc:.3f}")
print("gamma 太小→映射近线性(分不开); 合适 gamma→两环线性可分; 太大→过度局部化")


<a id="4"></a>
## 4. 缺点 / Limitations

- **核矩阵 $O(n^2)$ 存储 + $O(n^3)$ 分解** → 大数据不可行(可用 Nyström 近似)。
- **无显式逆变换**: 不像 PCA 能直接重构(有 `fit_inverse_transform` 但是近似/需额外学习)。
- **要调核与 gamma**, 没有"解释方差"那样直接的维度选择。
- 它和**谱聚类(6.7)**数学上近亲(都对核/相似度矩阵做特征分解), 也和后面的流形学习相关。


In [ ]:
from sklearn.decomposition import KernelPCA
# 不同核对比 / different kernels on circles
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, kern in zip(axes, ["linear", "poly", "rbf"]):
    kw = {"gamma":10} if kern!="linear" else {}
    if kern=="poly": kw={"degree":3, "gamma":1}
    z = KernelPCA(2, kernel=kern, **kw).fit_transform(X)
    ax.scatter(z[:,0], z[:,1], c=y, cmap="coolwarm", s=12)
    ax.set_title(f"kernel={kern}")
plt.suptitle("不同核: linear=普通PCA(分不开); poly/rbf 可非线性展开")
plt.tight_layout(); plt.show()
print("linear 核 = 普通 PCA; rbf 最常用于非线性降维")


<a id="5"></a>
## 5. 小结 / Summary

```
核 PCA: 用核技巧在隐式高维空间做 PCA = 原空间非线性降维
等价于对核矩阵 K(Gram) 做特征分解; 只用内积 K(xᵢ,xⱼ), 不显式映射
核矩阵必须中心化(无法显式减特征空间均值, 用 K 的等价中心化, sklearn 自动)
能展开同心圆/瑞士卷等线性 PCA 搞不定的非线性结构
缺点: O(n²)存/O(n³)分解不可扩展; 无直接逆变换; 需调核+gamma
与谱聚类(6.7)同源(都对核/相似度矩阵做谱分解)
```

### 💡 面试速查
1. **核 PCA = 核技巧 + PCA**: 隐式高维 PCA = 非线性降维
2. **对核矩阵 K 做特征分解**, 只需内积, 不显式映射(同 SVM)
3. **核矩阵中心化**是关键步骤(无法显式减特征空间均值)
4. **缺点**: O(n²/n³) 不可扩展, 无直接重构, 需调 gamma
5. 与**谱聚类**数学近亲; linear 核退化成普通 PCA

### 下一节
**6.10 因子分析**——和 PCA 长得像但模型不同: 假设观测由少数**潜在因子**加**特有噪声**生成, 是一个概率生成模型(心理测量学起源)。
